# All necessary imports for the analysis

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

## IMPORTANT: Update Data Path\n\nPlease update the `file_path` variable in the next cell to point to your actual data file.

# IMPORTANT: Please replace the placeholder with the actual path to your data file.\nfile_path = \PATH_TO_YOUR_DATA_FILE.csv\
df = pd.read_csv(file_path)

In [ ]:
df = pd.read_csv('data/OSF_1109_all_sample_3M_1charR.csv')

# Data Preparation

In [ ]:
df['Treat'] = (df['Dist_to_build'] <= 50000).astype(int)
df['event_date'] = pd.to_datetime(df['event_date'])
df['Month'] = pd.to_datetime(df['Month'])
df['Post'] = (df['Month'] >= df['event_date']).astype(int)

# 7.1. Exploratory Data Visualization
Before diving into the model, let's visualize our key variables to better understand the data distribution and identify any potential outliers or patterns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(df['Primary_Visits'], bins=50, ax=axes[0], kde=True)
axes[0].set_title("Distribution of Primary Visits")
sns.histplot(df['Emergency_Visits'], bins=50, ax=axes[1], kde=True, color='salmon')
axes[1].set_title("Distribution of Emergency Visits")
plt.suptitle("Visit Distributions")
plt.show()
monthly_visits = df.groupby('Month')[['Primary_Visits', 'Emergency_Visits']].mean()
monthly_visits.plot(figsize=(14, 7), subplots=True, layout=(2,1), title="Average Visits Per Month")
plt.xlabel("Month")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# 8.1. Visual Pre-Trend Checks for DiD Model
It is crucial to test the parallel trends assumption by plotting the outcome for both groups in the pre-treatment period.

In [ ]:
pre_treatment_df = df[df['Month'] < df['event_date']].copy()
pre_trend_data = pre_treatment_df.groupby(['Month', 'Treat'])[['Primary_Visits', 'Emergency_Visits']].mean().reset_index()
plt.figure(figsize=(14, 7))
sns.lineplot(data=pre_trend_data, x='Month', y='Primary_Visits', hue='Treat', marker='o')
plt.title("Pre-Treatment Trends in Primary Visits (Treatment vs. Control)")
plt.legend(title='Group', labels=['Control', 'Treatment'])
plt.grid(True)
plt.show()

# Main Difference-in-Differences Model

In [ ]:
formula = 'Primary_Visits ~ Treat * Post + C(Month) + C(zip_code)'
res_did = smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['zip_code']})
print(res_did.summary())

# 9.1. Formal Diagnostic Tests for Regression Models
After estimating the DiD model, we must perform diagnostic tests to validate the assumptions of the underlying linear regression.

In [ ]:
import statsmodels.api as sm
import statsmodels.stats.api as sms
import scipy.stats as stats
fitted_vals = res_did.fittedvalues
residuals = res_did.resid
sns.residplot(x=fitted_vals, y=residuals, lowess=True, line_kws={'color': 'red'})
plt.title("Residuals vs. Fitted Values")
plt.show()
bp_test = sms.het_breuschpagan(residuals, res_did.model.exog)
print(f"Breusch-Pagan Test p-value: {bp_test[1]:.4f}")
sm.qqplot(residuals, stats.t, fit=True, line='45')
plt.title("Q-Q Plot of Model Residuals")
plt.show()

# 10.1. Perform Robustness Checks

In [ ]:
print("--- Robustness Check: DiD Model with 'Emergency_Visits' as Outcome ---")
formula_robust = 'Emergency_Visits ~ Treat * Post + C(Month) + C(zip_code)'
res_did_robust = smf.ols(formula_robust, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['zip_code']})
print(res_did_robust.summary())

In [ ]:
print("--- Robustness Check: Placebo Test ---")
df_placebo = df.copy()
placebo_event_date = pd.to_datetime(df_placebo['event_date']).median() - pd.DateOffset(years=1)
df_placebo['Post_Placebo'] = (pd.to_datetime(df_placebo['Month']) >= placebo_event_date).astype(int)
formula_placebo = 'Primary_Visits ~ Treat * Post_Placebo + C(Month) + C(zip_code)'
res_did_placebo = smf.ols(formula_placebo, data=df_placebo).fit(cov_type='cluster', cov_kwds={'groups': df_placebo['zip_code']})
print(res_did_placebo.summary())

# 11.1. Visualizing the Main Difference-in-Differences Effect

In [ ]:
plot_data = df.groupby(['Month', 'Treat'])['Primary_Visits'].mean().reset_index()
avg_event_date = pd.to_datetime(df['event_date']).mean()
plt.figure(figsize=(14, 8))
ax = sns.lineplot(data=plot_data, x='Month', y='Primary_Visits', hue='Treat', style='Treat', markers=True, dashes=False)
ax.axvline(x=avg_event_date, color='r', linestyle='--', lw=2, label='Avg. Event Date')
ax.set_title('Average Primary Visits Over Time (Treatment vs. Control)')
ax.legend(title='Group')
plt.show()